In [2]:
import json
from pathlib import Path

In [3]:
def read_json(path: Path):
    with open(path, "r") as f:
        data = json.load(f)
    return data

In [4]:
tgt_path = Path(
    # "/Users/matthewho/Documents/research/ctx_editor/outputs/context_edit_gpt-5-mini_all/2026-01-29_09-00-12/traces/math/context_edit/sharded-GSM8K_1027.json"
    "/Users/matthewho/Documents/research/ctx_editor/outputs/baseline_gpt-5-mini_all/2026-01-29_08-03-51/traces/math/baseline/sharded-GSM8K_543.json"
)
ex_trace_data = read_json(tgt_path)

In [5]:
ex_trace_data

{'sample_id': 'sharded-GSM8K/543',
 'task_name': 'math',
 'experiment_type': 'baseline',
 'is_correct': False,
 'score': 0.0,
 'models': {'assistant': 'gpt-5-mini', 'user': 'unknown', 'system': 'unknown'},
 'trace': {'messages': [{'role': 'system',
    'content': 'As an expert problem solver solve step by step the following mathematical questions.',
    'timestamp': '2026-01-29 08:03:52'},
   {'role': 'user',
    'content': 'how much more money does Jairus earn than Jenny?',
    'timestamp': '2026-01-29 08:03:52',
    'cost_usd': 0.0},
   {'role': 'assistant',
    'content': 'I don’t have the numbers or the full problem — can you paste the exact question or the amounts/rates/hours? \n\nMeanwhile, here’s the general step‑by‑step method and a quick example so you know what I’ll do once you give the data.\n\nGeneral method\n1. Identify how much Jairus earns in total (call it J). This might be a product (rate × time), a fixed amount, or computed from given information.\n2. Identify how muc

In [7]:
REPO_DIR = Path("/Users/matthewho/Documents/research/ctx_editor")
DATA_DIR = REPO_DIR / "data" 
mini_eval_path = DATA_DIR / "lic_eval_subset.json"
mini_eval_data = read_json(mini_eval_path)

In [8]:
mini_eval_dict = {item["task_id"]: item for item in mini_eval_data}

In [10]:
mini_eval_dict["sharded-GSM8K/543"]

{'question': 'For each small task accomplished, Jairus gets $0.8 while Jenny gets $0.5. If each of them finished 20 tasks, how much more will Jairus get than Jenny?',
 'answer': 'The difference of the amount received by Jairus and Jenny is $0.8/task - $0.5/task = $<<0.8-0.5=0.3>>0.3/task.\nSo, Jairus will get $0.3/task x 20 tasks = $<<0.3*20=6>>6 more than Jenny.\n#### 6',
 'task_id': 'sharded-GSM8K/543',
 'shards': [{'shard_id': 1,
   'shard': 'how much more money does Jairus earn than Jenny?'},
  {'shard_id': 2,
   'shard': 'Jairus earns $0.8 for every small task he completes'},
  {'shard_id': 3, 'shard': 'Jenny earns $0.5 for each task she finishes'},
  {'shard_id': 4, 'shard': 'both Jairus and Jenny completed 20 tasks'}],
 'task': 'math',
 'full_spec_q': 'For each small task accomplished, Jairus gets $0.8 while Jenny gets $0.5. If each of them finished 20 tasks, how much more will Jairus get than Jenny?',
 'ground_truth_a': 'The difference of the amount received by Jairus and Jenny

In [15]:
import re
from typing import Any, Dict

In [22]:
def math_evaluator_function(extracted_answer: str, sample: Dict[str, Any]) -> bool:
    regexes_to_ignore = [",", "\\$", "(?s).*#### ", "\\.$"]

    # ground truth
    gold = sample["answer"].split("####")[1].strip().lower()

    try:
        # https://github.com/EleutherAI/lm-evaluation-harness/blob/bb098f13b05e361f01a5afe7b612779ce362b3f2/lm_eval/tasks/gsm8k/gsm8k.yaml#L42
        extracted_answer = extracted_answer.strip()
        print(f"Raw extracted answer: {repr(extracted_answer)}")
        # strict
        # extracted_answer = re.findall(r"(\-?[0-9\.\,]+)", extracted_answer)[0]
        # flexible
        extracted_answer = re.findall(r"(-?[$0-9.,]{2,})|(-?[0-9]+)", extracted_answer)[-1]
        print(f"Matched groups: {extracted_answer}")
        extracted_answer = [m for m in extracted_answer if m][0]
        print(f"Final extracted answer: {repr(extracted_answer)}")
    except:
        return {
            "score": 0.0,
            "error": f"Answer could not be extracted: {repr(extracted_answer)}",
        }

    # custom formatting fix
    # if dollar mark is in the answer, check for the cents and trim if necessary
    if re.search(r"\$", extracted_answer) and extracted_answer.endswith(".00"):
        extracted_answer = extracted_answer.rstrip(".00")
        print(f"Trimmed extracted answer cents: {repr(extracted_answer)}")

    # ref: https://github.com/EleutherAI/lm-evaluation-harness/blob/52df63b7b30da53c481ed9090598d9189fab1d91/lm_eval/api/metrics.py#L198
    # further normalize $ and , for both extracted_answer and gold
    for regex in regexes_to_ignore:
        extracted_answer = re.sub(regex, "", extracted_answer)
        print(f"Normalized extracted answer with regex {regex}: {repr(extracted_answer)}")
        gold = re.sub(regex, "", gold)
    print(f"Final normalized extracted answer: {repr(extracted_answer)}")
    score = 1.0 if extracted_answer == gold else 0.0
    return {"score": score}

In [ ]:
math_evaluator_function(
    "6.00",
    mini_eval_dict["sharded-GSM8K/543"]
)

Raw extracted answer: '6.00'
Matched groups: ('6.00', '')
Final extracted answer: '6.00'
Normalized extracted answer with regex ,: '6.00'
Normalized extracted answer with regex \$: '6.00'
Normalized extracted answer with regex (?s).*#### : '6.00'
Normalized extracted answer with regex \.$: '6.00'
Final normalized extracted answer: '6.00'


{'score': 0.0}